In [1]:
import numpy as np
import pandas as pd


In [2]:
train_df=pd.read_csv('IMDB_Dataset.csv')

In [3]:
train_df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [4]:
train_df[:5]

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
train_df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
print(train_df.shape)

(50000, 2)


In [7]:
train_df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [8]:
train_df['sentiment']

0        positive
1        positive
2        positive
3        negative
4        positive
           ...   
49995    positive
49996    negative
49997    negative
49998    negative
49999    negative
Name: sentiment, Length: 50000, dtype: str

In [9]:
X=train_df['review']
y=train_df['sentiment']

In [10]:
y

0        positive
1        positive
2        positive
3        negative
4        positive
           ...   
49995    positive
49996    negative
49997    negative
49998    negative
49999    negative
Name: sentiment, Length: 50000, dtype: str

In [11]:
y = y.map({'positive': 1, 'negative': 0})

In [12]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 50000, dtype: int64

In [13]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [14]:
X_train.shape

(40000,)

In [15]:
X_train.iloc[0].split

<function str.split(sep=None, maxsplit=-1)>

In [31]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [38]:
X_train_tokens = X_train.apply(
    lambda x: word_tokenize(x.lower())
)

X_test_tokens = X_test.apply(
    lambda x: word_tokenize(x.lower())
)

In [39]:
import string

X_train_tokens = X_train_tokens.apply(
    lambda tokens: [word for word in tokens if word not in string.punctuation]
)

X_test_tokens = X_test_tokens.apply(
    lambda tokens: [word for word in tokens if word not in string.punctuation]
)

In [40]:
print(X_train_tokens.iloc[0][:30])

['i', 'caught', 'this', 'little', 'gem', 'totally', 'by', 'accident', 'back', 'in', '1980', 'or', "'81", 'i', 'was', 'at', 'a', 'revival', 'theatre', 'to', 'see', 'two', 'old', 'silly', 'sci-fi', 'movies', 'the', 'theatre', 'was', 'packed']


In [41]:
print(X_train_tokens.iloc[0][:11])
print(len(X_train_tokens.iloc[0]))

['i', 'caught', 'this', 'little', 'gem', 'totally', 'by', 'accident', 'back', 'in', '1980']
174


In [42]:
from gensim.models import Word2Vec
model_cbow=Word2Vec(
    X_train_tokens,
    vector_size=100,
    window=2,
    min_count=1,
    workers=4,
    sg=0
)


In [43]:
print(len(model_cbow.wv))
print(len(model_cbow.wv['movie']))
print(model_cbow.wv.most_similar('movie'))

145247
100
[('film', 0.9602670669555664), ('flick', 0.7850455045700073), ('show', 0.7844347953796387), ('movie.', 0.7675447463989258), ('documentary', 0.756909966468811), ('movies', 0.7108225226402283), ('it', 0.7081096172332764), ('sequel', 0.7070795893669128), ('series', 0.702570915222168), ('picture', 0.7023977041244507)]


In [62]:
from gensim.models import Word2Vec
model_skipgram=Word2Vec(
    X_train_tokens,
    vector_size=100,
    window=2,
    min_count=5,
    workers=4,
    sg=1
)

In [63]:
print(len(model_skipgram.wv))
print(len(model_skipgram.wv['movie']))
print(model_skipgram.wv.most_similar('movie',topn=10))

40140
100
[('film', 0.9364815950393677), ('movie.', 0.824597954750061), ('flick', 0.8075257539749146), ('monstrosity', 0.7937099933624268), ('film-', 0.7801750898361206), ("'film", 0.768139123916626), ('film.', 0.7551521062850952), ("'movie", 0.7498212456703186), ('it.it', 0.7402060031890869), ('movie-', 0.7386699318885803)]


In [64]:
print(model_cbow.wv.most_similar('movie',topn=10))
print(model_skipgram.wv.most_similar('movie',topn=10))


[('film', 0.9602670669555664), ('flick', 0.7850455045700073), ('show', 0.7844347953796387), ('movie.', 0.7675447463989258), ('documentary', 0.756909966468811), ('movies', 0.7108225226402283), ('it', 0.7081096172332764), ('sequel', 0.7070795893669128), ('series', 0.702570915222168), ('picture', 0.7023977041244507)]
[('film', 0.9364815950393677), ('movie.', 0.824597954750061), ('flick', 0.8075257539749146), ('monstrosity', 0.7937099933624268), ('film-', 0.7801750898361206), ("'film", 0.768139123916626), ('film.', 0.7551521062850952), ("'movie", 0.7498212456703186), ('it.it', 0.7402060031890869), ('movie-', 0.7386699318885803)]


In [65]:
def avgword2vec(sentences,model):
    vector=[]
    for word in sentences:
        if word in model.wv:
            vector.append(model.wv[word])
    if len(sentences)==0:
        return np.zeros(model.vector_size)
    sentences_token=np.mean(vector,axis=0)
    return sentences_token

In [66]:
result = avgword2vec(X_train_tokens.iloc[0], model_cbow)

print(result)
print(result.shape)

[ 0.80341953  0.41137034  0.32452834 -0.0948631   0.07846178 -0.6498688
  0.19867781 -0.01291704  0.3166148  -0.1311479   0.01370993 -0.19170739
  0.49472755  0.87817645 -0.6579515  -0.78664905 -0.49489465  0.02716954
 -0.7475828  -0.4094721  -0.07031792 -0.24443565 -0.6085438  -0.46797192
  0.0547068  -0.5425957   0.18474478 -0.5479269  -0.13525069  0.39887664
 -0.29526126  0.21266142  0.6475229  -0.53790236  0.62954706  1.1329205
 -0.7349     -0.5548923  -0.6583657  -0.99529696  0.63299185 -0.6743356
 -0.11549822  0.28915876 -0.02370836  0.09048206 -0.53734064 -0.12005536
 -0.37921405 -0.09322222  0.4181526  -0.4604988  -0.49497947  0.637021
 -0.0862475   0.35981694 -0.21666132 -0.93131864 -0.30784386  0.6352763
 -0.33663943 -0.2458798  -0.8852444   0.06090458  0.70072687  0.32937106
  0.33566618 -0.27886552 -0.17822723 -0.35053185 -0.44318363 -0.28863528
  0.08004175  0.27261457  0.7278357  -0.37036702  0.59104943  0.15088104
  0.46072528 -0.12247584  0.1898572   0.50853634  0.59487

In [50]:
X_train_avg=X_train_tokens.apply(
    lambda x:avgword2vec(x,model_cbow)
)
X_test_avg=X_test_tokens.apply(
    lambda x:avgword2vec(x,model_cbow)
)

In [51]:
print(X_train_avg.shape)
print(X_test_avg.shape)

(40000,)
(10000,)


In [52]:
X_train_avg=np.vstack(X_train_avg)
X_test_avg=np.vstack(X_test_avg)
print(X_train_avg.shape)
print(X_test_avg.shape)

(40000, 100)
(10000, 100)


In [53]:
from sklearn.linear_model import LogisticRegression
model_Lg=LogisticRegression(max_iter=1000)
model_Lg.fit(X_train_avg,y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [54]:
y_pred=model_Lg.predict(X_test_avg)

In [55]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.8302
              precision    recall  f1-score   support

           0       0.83      0.82      0.83      5000
           1       0.83      0.84      0.83      5000

    accuracy                           0.83     10000
   macro avg       0.83      0.83      0.83     10000
weighted avg       0.83      0.83      0.83     10000



In [56]:
print(X_train_avg.shape)
print(X_test_avg.shape)
print(y_train.shape)
print(y_test.shape)

(40000, 100)
(10000, 100)
(40000,)
(10000,)


In [67]:
X_train_avg=X_train_tokens.apply(
    lambda x:avgword2vec(x,model_skipgram)
)
X_test_avg=X_test_tokens.apply(
    lambda x:avgword2vec(x,model_skipgram)
)

In [68]:
X_train_avg=np.vstack(X_train_avg)
X_test_avg=np.vstack(X_test_avg)
print(X_train_avg.shape)
print(X_test_avg.shape)

(40000, 100)
(10000, 100)


In [69]:
model_Lg.fit(X_train_avg,y_train)


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [70]:
y_pred=model_Lg.predict(X_test_avg)

In [71]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.847
              precision    recall  f1-score   support

           0       0.85      0.84      0.85      5000
           1       0.85      0.85      0.85      5000

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



In [75]:
import pickle


model_skipgram.save("word2vec_skipgram.model")


with open("sentiment_model.pkl", "wb") as f:
    pickle.dump(model_Lg, f)


label_mapping = {
    "negative": 0,
    "positive": 1
}

with open("label_mapping.pkl", "wb") as f:
    pickle.dump(label_mapping, f)